# 结构化生成

## 动机

当我们用 LLM 回答问题的时候，输出是自由格式的文本。但在很多真实应用里，我们需要特定格式的输出：一个数字、一个 JSON 对象、一个日期、一个列表中的选项等等。通常的变通方法是生成自由文本，然后用正则表达式解析它：


In [ ]:
import re

answer = """The first 10 digits of pi (π) are as follows:

3.1415926535
"""

regex = r"([0-9]+)?\.[0-9]+"
print(re.search(regex, answer))

这能工作，但很脆弱：LLM 可能产生完全不匹配模式的文本，迫使我们重试或失败。**结构化生成**通过在_生成过程中_约束 LLM 来解决这个问题，保证每个输出都匹配想要的格式。

## 本实操的目标

在这个实操里，你将从头实现结构化生成，依次推进三种效率越来越高的方法：

1. **朴素方法** -- 每一步生成时，用正则的部分匹配去试词表中的每个 token。简单，但每步 $O(V)$。
2. **基于 DFA 的方法** -- 把正则编译成确定性有限自动机（DFA），然后预先计算从每个 DFA 状态出发哪些 token 是有效的。每步代价降到 $O(1)$。
3. **合并（Coalescence）** -- 注意到很多 DFA 状态允许完全相同的 token 集合，所以我们可以预先计算并在等价状态之间_共享_掩码数组，甚至消除数组分配的开销。

我们遵循 Brandon T. Willard 和 Remi Louf 的 [Efficient Guided Generation for Large Language Models](https://arxiv.org/abs/2307.09702) 的思路。

## 准备工作

整个 notebook 我们使用一个只有 4 个 token 的玩具词表：`["a", ".", ".2", "1"]`，以及正则模式 `([0-9]+)?\.[0-9]+`（一个像 `3.14` 这样的小数）。这样规模足够小，可以手工检查。最后，我们会放大到 GPT-2 真实的 5 万 token 词表。


## 第 1 部分：用正则部分匹配做朴素受限生成

关键思想很简单：每一步生成时，我们把会让输出与目标正则不兼容的 token **屏蔽掉**。LLM 只能从仍然可能产生有效匹配的 token 中采样。

具体来说，算法这样工作：

1. 以空字符串作为前缀开始。
2. 对词表中的每个 token，把它拼到前缀后面，检查结果是否是正则的**部分匹配**（即，如果我们继续追加字符，仍然可能产生完整匹配）。
3. 如果不是，把该 token 的 logit 设为 $-\infty$，使它不能被采样。
4. 从屏蔽后的 logits 中采样下一个 token。
5. 把采样的 token 追加到前缀，回到第 2 步。

下图演示了正则 `[0-9]+\.[0-9]`、词表 `["a", ".", ".2", "1"]` 的这个过程：

![](https://cdn.prod.website-files.com/665725b00d910f65bec567fc/668c29d45780ee71a367c839_naive.png)

### 部分匹配

**部分匹配**是指一个字符串匹配正则直到它的最后一个字符 -- 也就是我们能为它找到一个能产生完整匹配的续写。Python 标准的 `re` 库不支持部分匹配，但 [regex](https://github.com/mrabarnett/mrab-regex) 库支持，通过 `partial=True` 标志：


In [ ]:
import regex as re

regex = r"([0-9]+)?\.[0-9]+"
print(re.fullmatch(regex, '.2', partial=True))

In [ ]:
print(re.fullmatch(regex, '1.2', partial=True))

In [ ]:
print(re.fullmatch(regex, '1.2a', partial=True))

### 练习 1：实现朴素受限生成

用 `re.fullmatch(regex, string, partial=True)` 在每一步生成时构建一个 logit 掩码。掩码应该是一个 numpy 数组：对"与当前前缀拼接后得到部分匹配"的 token 为 `0`，否则为 `-math.inf`。我们用均匀 logits（全为 1）模拟一个 LLM，这样生成完全由掩码驱动。


In [ ]:
import math
import regex as re
import numpy as np

from scipy.special import softmax

np.random.seed(12349)

logits = np.array([1., 1., 1., 1.])  # 随机模型，概率相等
vocabulary = ["a", ".", ".2", "1"]

regex = r"([0-9]+)?\.[0-9]+"

completion = ""
for _ in range(7):

    # 构建 logit 掩码
    # 对词表中的每个 token，检查把它追加到当前
    # 补全后面是否可能导致有效匹配（用部分匹配）。
    # 有效 token 的掩码设为 0，无效的设为 -inf。
    # 结果应该是一个叫 `mask` 的 numpy 数组。
    #
    # 你的代码
    #
    
    masked_logits = logits + mask

    # 采样下一个 token
    probs = softmax(masked_logits)
    next_token_id = np.random.choice(len(vocabulary), p=probs)

    completion += vocabulary[next_token_id]

print(completion)

## 第 2 部分：基于 DFA 的受限生成

朴素方法能正确工作，但对真实的 LLM 来说**太慢了**。词表有 $V \approx 50{,}000$ 个 token，我们_每生成一个 token_ 都要做 5 万次正则部分匹配。在 Python 里这很容易主导推理时间。

关键洞察是：正则表达式等价于**确定性有限自动机（DFA）**。DFA 是一个有向图，其中：
- 每个**节点**是一个状态。
- 每条**边**是一个带字符（或字符类）标签的转移。
- 有一个**初始状态**和一个或多个**接受（最终）状态**。

要检查一个字符串是否匹配，你一个字符一个字符地在 DFA 上走：

1. 从初始状态和完整字符串开始。
2. 读取下一个字符。如果有匹配的转移，就沿着它走到下一个状态。否则，**拒绝**。
3. 消费完整个字符串后，如果你在一个最终状态，就**接受**。

### 为什么这有帮助？

与其每一步对每个 `(前缀 + token)` 对运行一次正则引擎，我们可以：
1. **预先计算**每个 DFA 状态哪些 token 能产生有效转移。
2. 生成时，只需要**查表**得到当前状态允许的 token 集合 -- 不再需要正则匹配。

这把每步 $O(V)$ 的代价换成了一次性的 $O(V \times |\text{states}|)$ 预计算。

### 构建 DFA

我们用 [interegular](https://github.com/MegaIng/interegular) 库把正则转换成等价的 DFA。看看它长什么样：


In [ ]:
import interegular

regex = r"([0-9]+)?\.[0-9]+"
fsm = interegular.parse_pattern(regex).to_fsm()

print(fsm)

In [ ]:
print(fsm.alphabet)

In [ ]:
for (start, transitions) in fsm.map.items():
    print(start, transitions)

In [ ]:
print(fsm.initial)

In [ ]:
print(fsm.finals)

In [ ]:
fsm.alphabet.keys()

### 解读 DFA

DFA 有三个关键组成部分：

- **`fsm.alphabet`** 把每个字符映射到一个**符号索引**。行为相同的字符（比如所有数字 `0`-`9`）共享同一个索引。还有一个 `anything_else` 符号，用于没有显式列出的字符。
- **`fsm.map`** 是转移表：`fsm.map[state][symbol_index] = next_state`。如果某个 `(state, symbol_index)` 对缺失，就没有有效转移（字符串被拒绝）。
- **`fsm.initial`** 和 **`fsm.finals`** 分别是起始状态和接受状态集合。

我们把这个 DFA 可视化成一张图：


In [ ]:
from interegular import fsm as fsm_module
from collections import defaultdict

# 从 fsm.alphabet 动态构建边的标签
idx_to_chars = defaultdict(list)
for char, idx in fsm.alphabet.items():
    if char is fsm_module.anything_else:
        idx_to_chars[idx].append("*")
    else:
        idx_to_chars[idx].append(char)

# 把分组折叠成可读标签，例如 ['0','1',...,'9'] -> "[0-9]"
idx_to_label = {}
for idx, chars in idx_to_chars.items():
    if len(chars) > 3:
        idx_to_label[idx] = f"[{chars[0]}-{chars[-1]}]"
    else:
        idx_to_label[idx] = ",".join(chars)

In [ ]:
import graphviz
from IPython.display import display

# 定义正则模式
regex = r"([0-9]+)?\.[0-9]+"

# 把正则转换成一个有限状态机（FSM）
fsm = interegular.parse_pattern(regex).to_fsm()

# 生成 Graphviz DOT 格式表示
dot = graphviz.Digraph(format="png")

# 往图里添加状态
for state in fsm.states:
    shape = "doublecircle" if state in fsm.finals else "circle"
    dot.node(str(state), shape=shape)

# 往图里添加转移
for (start, transitions) in fsm.map.items():
    for char, end in transitions.items():
        dot.edge(str(start), str(end), label=idx_to_label.get(char, str(char)))

display(dot)

In [ ]:
fsm.map

### 从字符到 token：在 DFA 上行走

DFA 作用在**字符**上，但 LLM 生成的是**token**（可能是多字符的字符串，比如 `".2"` 或 `"1"`）。要检查一个 token 是否与给定的 DFA 状态兼容，我们需要在 DFA 上一个字符一个字符地"走"这个 token。

给定一个起始状态和一个 token 字符串，我们：
1. 在 `fsm.alphabet` 里查第一个字符，得到它的符号索引。
2. 检查当前状态对这个符号是否有转移。如果没有，token 从该状态被**拒绝**。
3. 沿着转移走到下一个状态，对剩余字符重复。
4. 如果我们没有遭到拒绝就消费完所有字符，那么 token 从该状态是**有效**的。

例如，从状态 0 出发，token `".2"`：
- 字符 `"."` 的符号索引是 2，`fsm.map[0][2] = 2` $\Rightarrow$ 移到状态 2。
- 字符 `"2"` 的符号索引是 0（数字），`fsm.map[2][0] = 4` $\Rightarrow$ 移到状态 4。
- 我们经过了状态 $(0, 2, 4)$。状态 4 是最终状态，所以 `".2"` 是一个完整的有效匹配！

反过来，从状态 0 出发的 token `"a"`：字符 `"a"` 映射到 `anything_else`（符号索引 1），且没有转移 `fsm.map[0][1]`，所以 `"a"` 被拒绝。

### 练习 2：实现 `partial_match`

写一个函数：在 DFA 上行走一个 token，返回走过的状态元组；如果 token 被拒绝则返回 `None`。


In [ ]:
def partial_match(state, token):
    """Partially match the token to the DFA starting from `state`.

    We iterate over the token's symbols, and at each step transition to the 
    next state if we find a valid transition. 
    If there is a stage without a valid transision, we return None, otherwise
    we return a tuple that contains the sequence of traversed states.

    Hints:
    - Use fsm.alphabet[symbol] to get the alphabet index of a character.
    - Use fsm.map[state] to get the transitions from a state.
    - Return a tuple of all traversed states (including the starting state).
    """
    
    traversed_states = (state,)
    # 遍历 token 的符号，每一步尝试转移
    # 到一个新的 DFA 状态。
    #
    # 你的代码
    #
    
    return traversed_states

In [ ]:
token = ".21"
print(partial_match(0, token))

In [ ]:
token = ".21."
print(partial_match(0, token))

### 练习 3：构建状态到 token 的索引

现在我们需要预先计算：对每个 DFA 状态，词表中哪些 token 对应有效的转移。我们构建两个数据结构：

- **`states_to_vocab[state]`**：从 `state` 出发有效的 token ID 集合。
- **`states_token_states[state][token_id]`**：从 `state` 消费完这个 token 后落入的 DFA 状态。

这是一次性的预计算，它让生成变快：不再每一步对每个 token 运行正则，我们只需要查 `states_to_vocab[current_state]`。


In [ ]:
from collections import defaultdict

vocabulary = ["a", ".", ".2", "1"]

# 从 DFA 状态到对应有效转移的 token 的映射
# （从该状态出发）。
states_to_vocab = defaultdict(set)
states_token_states = defaultdict(dict)

# （一次性）遍历词表，对每个 token，从每个 DFA 状态
# 检查 partial_match 是否找到有效路径。
# 如果是，把 token_id 记录到 states_to_vocab[state]，把落点状态
# 记录到 states_token_states[state][token_id]。
#
# 你的代码
#

In [ ]:
states_to_vocab

### 练习 4：基于 DFA 的生成

有了索引，生成就变得直接：

1. 从初始 DFA 状态开始。查 `states_to_vocab[state]` 得到允许的 token。
2. 构建掩码：全为 $-\infty$，然后把允许的位置设为 0。
3. 把掩码加到 logits 上并采样。
4. 查 `states_token_states[state][token_id]` 转移到下一个 DFA 状态。
5. 重复。

你应该得到和朴素方法**相同的结果**（`11.21111`），因为随机种子相同。


In [ ]:
np.random.seed(12349)  # 你应该得到和之前一样的结果

logits = np.array([1., 1., 1., 1.])  # 和之前一样

regex = r"([0-9]+)?\.[0-9]+"

completion = ""
state = fsm.initial
for _ in range(7):

    # 用 states_to_vocab[state] 构建 logit 掩码
    # （不需要正则 -- 只是集合查找！）
    #
    # 你的代码
    #

print(completion)

### 一个小基准测试


我们现在在逐渐增大的词表上对两种方法（朴素正则部分匹配 vs 基于 DFA 的受限解码）做基准测试。这说明了为什么随着词表增长到真实的 LLM 规模（约 5 万 token），朴素方法变得不切实际，以及为什么通过 DFA 预先计算有效转移是必要的。

我们构建一个合成词表：一个小的"有用"核心（数字/点 token），其余用永远不会匹配的垃圾 token 填充（这是一个现实场景：大多数 token 与正则无关）。


In [ ]:
import time

REGEX = r"([0-9]+)?\.[0-9]+"

def build_vocabulary(size: int):
    """Build a synthetic vocabulary: a small useful core + junk padding tokens."""
    core = [
        ".", "0", "1", "2", "3", "4", "5", "6", "7", "8", "9",
        ".0", ".1", ".2", ".3", ".4", ".5", ".6", ".7", ".8", ".9",
        "10", "12", "42", "100", "256",
    ]
    padding = [f"tok_{i}" for i in range(size - len(core))]
    vocab = core + padding
    return vocab[:size]


def naive_mask(completion, vocabulary, pattern):
    """Approach 1: one regex partial match per token per step — O(V) per token."""
    mask = []
    for token in vocabulary:
        tentative = completion + token
        if re.fullmatch(pattern, tentative, partial=True) is None:
            mask.append(-math.inf)
        else:
            mask.append(0.0)
    return np.array(mask)


def build_dfa_index(vocabulary, pattern):
    """Approach 2: one-time precomputation — build the DFA and index valid tokens per state."""
    fsm = interegular.parse_pattern(pattern).to_fsm()

    def _partial_match(state, token):
        traversed = (state,)
        for symbol in token:
            alphabet_idx = fsm.alphabet.get(symbol)
            if alphabet_idx is None:
                alphabet_idx = fsm.alphabet.get(interegular.fsm.anything_else)
            if state not in fsm.map or alphabet_idx not in fsm.map[state]:
                return None
            state = fsm.map[state][alphabet_idx]
            traversed += (state,)
        return traversed

    states_to_vocab = defaultdict(set)
    states_token_states = defaultdict(dict)

    for token_id, token in enumerate(vocabulary):
        for state in fsm.map:
            path = _partial_match(state, token)
            if path is not None:
                states_to_vocab[state].add(token_id)
                states_token_states[state][token_id] = path[-1]

    return fsm, states_to_vocab, states_token_states


def dfa_mask(state, states_to_vocab, vocab_size):
    """O(1) lookup per token using the precomputed index."""
    mask = np.full(vocab_size, -np.inf)
    valid = list(states_to_vocab[state])
    if valid:
        mask[valid] = 0.0
    return mask

In [ ]:
def benchmark(vocab_sizes, n_steps=7, n_repeats=3):
    results = []

    print(f"Regex: {REGEX}")
    print(f"Generating {n_steps} tokens per run, median of {n_repeats} repeats\n")
    print(f"{'Vocab size':>12}  {'Naive (ms/step)':>16}  {'DFA mask (ms/step)':>18}  {'DFA precomp (ms)':>16}  {'Speedup':>8}")
    print("-" * 80)

    for V in vocab_sizes:
        vocabulary = build_vocabulary(V)
        logits = np.ones(V)

        # ---- 朴素方法计时 ----
        naive_times = []
        for _ in range(n_repeats):
            np.random.seed(42)
            completion = ""
            t0 = time.perf_counter()
            for _ in range(n_steps):
                mask = naive_mask(completion, vocabulary, REGEX)
                masked_logits = logits + mask
                probs = softmax(masked_logits)
                next_id = np.random.choice(V, p=probs)
                completion += vocabulary[next_id]
            naive_times.append((time.perf_counter() - t0) / n_steps)
        naive_ms = np.median(naive_times) * 1000

        # ---- DFA 预计算 ----
        t0 = time.perf_counter()
        fsm_bench, s2v, sts = build_dfa_index(vocabulary, REGEX)
        precomp_ms = (time.perf_counter() - t0) * 1000

        # ---- DFA 掩码计时 ----
        dfa_times = []
        for _ in range(n_repeats):
            np.random.seed(42)
            state = fsm_bench.initial
            completion = ""
            t0 = time.perf_counter()
            for _ in range(n_steps):
                mask = dfa_mask(state, s2v, V)
                masked_logits = logits + mask
                probs = softmax(masked_logits)
                next_id = np.random.choice(V, p=probs)
                state = sts[state][next_id]
                completion += vocabulary[next_id]
            dfa_times.append((time.perf_counter() - t0) / n_steps)
        dfa_ms = np.median(dfa_times) * 1000

        speedup = naive_ms / dfa_ms if dfa_ms > 0 else float("inf")
        results.append((V, naive_ms, dfa_ms, precomp_ms, speedup))
        print(f"{V:>12,}  {naive_ms:>14.2f}ms  {dfa_ms:>16.3f}ms  {precomp_ms:>14.1f}ms  {speedup:>7.0f}x")

    return results

results = benchmark([100, 500, 1_000, 5_000, 10_000])

In [ ]:
import matplotlib.pyplot as plt

vocab_sizes = [r[0] for r in results]
naive_times = [r[1] for r in results]
dfa_times = [r[2] for r in results]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(vocab_sizes, naive_times, "o-", label="Naive (regex partial match)")
ax.plot(vocab_sizes, dfa_times, "s-", label="DFA (precomputed index)")
ax.set_xlabel("Vocabulary size")
ax.set_ylabel("Time per generation step (ms)")
ax.set_title("Constrained Decoding: Naive vs DFA")
ax.legend()
ax.set_yscale("log")
ax.grid(True, which="both", ls="--", alpha=0.5)
plt.tight_layout()
plt.show()

**要点：**
- **朴素：** 每生成一个 token 做 $O(V)$ 次部分匹配 -- 随词表大小线性增长，主导生成时间。
- **DFA：** $O(V \times |states|)$ 的**一次性**预计算，之后每 token $O(1)$ 查表。
- 在 $V=10k$ 以上，朴素方法慢几个数量级。在真实的 LLM 词表规模（约 5 万）下，它完全不可用。


## 合并（Coalescence）：预计算掩码

DFA 方法已经带来了巨大的加速：不再对每个 token 运行一次正则部分匹配，而是做集合查找。但每一步生成仍然有开销：我们分配一个 numpy 数组（`np.full(V, -inf)`），然后把有效索引设为 0。

**合并**观察到，很多 DFA 状态共享*完全相同*的允许 token 集合。对我们的正则，看看上面的 `states_to_vocab`：状态 4 和 5 都允许 `{3}`，状态 0、1、3 都允许 `{1, 2, 3}`。为什么对等价状态重新计算相同的掩码？

思路：
1. 构建 token 级 FSM（用 `token_fsm.py`）
2. 按它们的允许 token ID 的 `frozenset` 对状态分组
3. 在一次性的预处理中，**每个组预计算一个掩码数组**
4. 构建查找表：`state → precomputed_mask`
5. 生成时：`mask = precomputed_masks[state]` -- 一次字典查找，**零数组分配**


In [ ]:
from token_fsm import make_deterministic_fsm, create_fsm_index_tokenizer


def build_coalesced_index(vocabulary, pattern):
    """Build the token-level FSM and precompute one mask per unique allowed-token set.

    Returns
    -------
    token_fsm : TokenFSM with .map[state][token_id] -> next_state
    precomputed_masks : dict[state] -> np.array (the logit mask, ready to use)
    precomp_ms : total precomputation time in milliseconds
    """
    V = len(vocabulary)
    t0 = time.perf_counter()

    # 第 1 步：正则 -> 字符级 DFA -> 清理
    raw_fsm = interegular.parse_pattern(pattern).to_fsm()
    clean_fsm, _ = make_deterministic_fsm(raw_fsm)

    # 第 2 步：构建 token 级 FSM
    tok_fsm, index = create_fsm_index_tokenizer(clean_fsm, vocabulary)

    # 第 3 步：合并 -- 按允许的 token 集合对状态分组
    # 对 clean_fsm.states 中的每个状态，从 tok_fsm.map 获取
    # 允许 token ID 的 frozenset。允许集合相同的状态应该共享同一个
    # 预计算掩码（形状为 (V,) 的 numpy 数组，允许的 token 为 0，
    # 其余为 -inf）。
    # 构建：
    #   mask_cache: frozenset -> np.array（每个唯一 token 集合一个掩码）
    #   precomputed_masks: state -> np.array（生成时查找）
    #
    # 你的代码
    #

    precomp_ms = (time.perf_counter() - t0) * 1000

    n_states = len(clean_fsm.states)
    n_groups = len(mask_cache)
    print(f"  Coalescence: {n_states} states -> {n_groups} unique masks")

    return tok_fsm, precomputed_masks, precomp_ms

### 练习 5：实现合并

辅助模块 `token_fsm.py` 提供两个函数：
- **`make_deterministic_fsm(fsm)`** 清理字符级 DFA（把状态重映射成连续整数）。
- **`create_fsm_index_tokenizer(fsm, vocabulary)`** 构建一个 **token 级 FSM** -- 一个转移作用在 token ID 上而不是字符上的 FSM。它返回一个 `TokenFSM` 对象，带 `.map[state][token_id] = next_state` 转移表。

你的任务是实现合并这一步：把共享相同允许 token 集合的状态分组，每组预计算一个掩码，并构建 `precomputed_masks` 字典。


In [ ]:
# --- 配置 ---
regex_pattern = r"([0-9]+)?\.[0-9]+"
vocabulary = ["a", ".", ".2", "1"]

print(f"Regex:      {regex_pattern}")
print(f"Vocabulary: {vocabulary}")

### 逐步走一遍 token 级 FSM

在使用 `build_coalesced_index` 之前，让我们看看 `token_fsm.py` 的流水线在我们的玩具例子上是怎么工作的。我们逐步走一遍：解析正则、清理 DFA、构建 token 级 FSM，并用它运行受限生成。


In [ ]:
# --- 第 1 步：把正则解析成字符级 DFA ---
raw_fsm = interegular.parse_pattern(regex_pattern).to_fsm()
print("── Raw character-level DFA ──")
print(f"  States:  {raw_fsm.states}")
print(f"  Initial: {raw_fsm.initial}")
print(f"  Finals:  {raw_fsm.finals}")
print(f"  Transitions:")
for state, trans in sorted(raw_fsm.map.items(), key=lambda x: str(x[0])):
    print(f"    State {state}: {dict(trans)}")

In [ ]:
# --- 第 2 步：清理 DFA ---
clean_fsm, state_mapping = make_deterministic_fsm(raw_fsm)
print(f"── Cleaned DFA (state mapping: {state_mapping}) ──")
print(f"  States:  {clean_fsm.states}")
print(f"  Initial: {clean_fsm.initial}")
print(f"  Finals:  {clean_fsm.finals}")

In [ ]:
# --- 第 3 步：构建 token 级 FSM ---
token_fsm, index = create_fsm_index_tokenizer(clean_fsm, vocabulary)

print(f"── Token-level FSM ──")
print(f"  Initial state: {token_fsm.initial}")
print(f"  Accept states: {token_fsm.finals}")
print(f"\n  Transition table (state → token → next_state):")
for state in sorted(token_fsm.map.keys()):
    for tid, next_s in sorted(token_fsm.map[state].items()):
        print(f"    State {state} --[{tid}: '{vocabulary[tid]}']→ State {next_s}")

In [ ]:
# --- 第 4 步：用 token FSM 做受限生成 ---
print("── Constrained generation ──")

np.random.seed(12349)
logits = np.ones(len(vocabulary))

completion = ""
state = token_fsm.initial

for step in range(7):
    # 屏蔽现在只是一次集合查找 -- 不需要正则！
    allowed = token_fsm.allowed_token_ids(state)
    mask = np.full(len(vocabulary), -np.inf)
    mask[list(allowed)] = 0.0

    masked_logits = logits + mask
    probs = softmax(masked_logits)
    next_id = np.random.choice(len(vocabulary), p=probs)

    next_state = token_fsm.next_state(state, next_id)
    print(f"  Step {step}: state={state}, "
          f"allowed={[vocabulary[i] for i in sorted(allowed)]}, "
          f"sampled='{vocabulary[next_id]}' → state={next_state}")

    state = next_state
    completion += vocabulary[next_id]

print(f"\n  Final completion: '{completion}'")
is_full_match = state in token_fsm.finals
print(f"  In accept state?  {is_full_match}")

## 第 4 部分：把方法应用到带真实分词器的 JSON schema

到目前为止我们用的都是 4 个 token 的玩具词表。在实践中，LLM 使用**BPE（字节对编码）分词器**，约 5 万 token，单个 token 可能是多字符的子词，比如 `"name"`、`":"` 或 `"John"`。

这带来了一个重要的微妙之处：单个 BPE token 可以一次跨越**多个 DFA 转移**。例如，token `"name"` 一步走过 4 个字符级 DFA 状态。这正是为什么我们需要**token 级 FSM** 而不是字符级 FSM。

让我们看看 token 级 FSM 如何用 GPT-2 的真实分词器处理 JSON 结构的正则。这个模式强制一个特定的 JSON schema：

```
\{"name":("John"|"Paul"),"age":(20|30)\}
```

这是**结构化输出**（JSON）和**受限解码**之间的桥梁：我们把 schema 表示成正则，编译成 DFA，然后在真实的 BPE 词表上构建 token 级索引。

### 练习 6：实现 `walk_token_fsm` 并构建索引

这个练习和练习 2（`partial_match`）类似，但现在你还需要处理 `anything_else` 字母表符号（用于正则中没有显式列出的字符），并在 GPT-2 的 5 万词表上构建完整索引。


In [ ]:
json_pattern = r'\{"name":("John"|"Paul"),"age":(20|30)\}'

# 构建字符级 DFA
raw_fsm = interegular.parse_pattern(json_pattern).to_fsm()
json_fsm, _ = make_deterministic_fsm(raw_fsm)

print(f"Pattern: {json_pattern}")
print(f"DFA: {len(json_fsm.states)} states, {len(json_fsm.finals)} accept state(s)")
print(f"\nCharacter-level DFA transitions:")
for state in sorted(json_fsm.map.keys()):
    print(f"  State {state}: {dict(json_fsm.map[state])}")

In [ ]:
from transformers import AutoTokenizer
from interegular import fsm as fsm_module

tokenizer = AutoTokenizer.from_pretrained("gpt2")

def walk_token_fsm(fsm, state, token_str):
    """Walk a token string through the character-level DFA.
    
    For each character in token_str, look up its alphabet symbol index,
    then check if there's a valid transition from the current state.
    Return the final state if the full token is consumed, or None if
    any character has no valid transition.

    Hints:
    - Use fsm.alphabet to map characters to symbol indices.
    - Handle characters not in the alphabet using fsm_module.anything_else.
    - Use fsm.map.get(state, {}) for transitions.
    """
    #
    # 你的代码
    #

print(f"Tokenizer vocab size: {tokenizer.vocab_size:,}")
print("Building token-level FSM index (this may take a minute)...")

t0 = time.perf_counter()
json_index = defaultdict(dict)
# 对每个 DFA 状态和词表中的每个 token，用 walk_token_fsm 检查
# 从该状态出发 token 能否走过 DFA。
# 如果能，把落点状态记录到 json_index[state][token_id]。
# 用 tokenizer.decode([token_id]) 得到 token 字符串。
#
# 你的代码
#
elapsed = time.perf_counter() - t0

print(f"Index built in {elapsed:.1f}s")
print(f"States with valid transitions: {len(json_index)}")

In [ ]:
# 显示：对每个状态，展示解码后的 BPE token 及其目标状态
for state in sorted(json_index.keys()):
    transitions_decoded = {
        repr(tokenizer.decode([tid])): next_s
        for tid, next_s in json_index[state].items()
    }
    print(f"State {state}: {transitions_decoded}")

注意 BPE 的多字符 token（比如 `'name'`、`'":"'`、`'John'`、`',"'`）每一个都对应穿过 DFA 的有效多步转移。每一步生成时，LLM 可以通过发出单个多字符 token，一次跳过多个 DFA 状态。这正是为什么我们需要 **token 级** FSM 而不是字符级 FSM：真实分词器不会一个字符一个字符地输出。


In [ ]:
def plot_char_dfa(fsm):
    """Plot the character-level DFA with graphviz."""
    idx_to_chars = defaultdict(list)
    for char, idx in fsm.alphabet.items():
        if char is fsm_module.anything_else:
            idx_to_chars[idx].append("*")
        elif char == " ":
            idx_to_chars[idx].append("⎵")
        else:
            idx_to_chars[idx].append(char)

    dot = graphviz.Digraph(
        name="Character-level DFA",
        graph_attr={"rankdir": "LR", "dpi": "50", "fontsize": "12"},
        node_attr={"fontsize": "11"},
        edge_attr={"fontsize": "9"},
    )
    dot.node("start", shape="point", width="0")
    dot.edge("start", str(fsm.initial))

    for state in sorted(fsm.states):
        shape = "doublecircle" if state in fsm.finals else "circle"
        dot.node(str(state), str(state), shape=shape)

    edge_labels = defaultdict(list)
    for state, transitions in fsm.map.items():
        for sym_idx, target in transitions.items():
            chars = idx_to_chars.get(sym_idx, [f"[{sym_idx}]"])
            edge_labels[(str(state), str(target))].append(",".join(chars))

    for (src, dst), labels in edge_labels.items():
        dot.edge(src, dst, label=" | ".join(labels))
    return dot


def plot_token_fsm(index, vocabulary, initial, finals):
    """Plot the token-level FSM with BPE tokens as edge labels."""
    dot = graphviz.Digraph(
        name="Token-level FSM",
        graph_attr={"rankdir": "LR", "dpi": "45", "fontsize": "12"},
        node_attr={"fontsize": "11"},
        edge_attr={"fontsize": "9"},
    )
    all_states = {initial} | set(finals)
    for state, transitions in index.items():
        all_states.add(state)
        for tid, target in transitions.items():
            all_states.add(target)

    dot.node("start", shape="point", width="0")
    dot.edge("start", str(initial))

    for state in sorted(all_states):
        shape = "doublecircle" if state in finals else "circle"
        dot.node(str(state), str(state), shape=shape)

    edge_labels = defaultdict(list)
    for state, transitions in index.items():
        for tid, target in transitions.items():
            token_str = vocabulary[tid]
            disp = token_str.replace('"', '\\"').replace("{", "\\{").replace("}", "\\}")
            edge_labels[(str(state), str(target))].append(f'"{disp}"')

    for (src, dst), labels in edge_labels.items():
        dot.edge(src, dst, label=" | ".join(labels))
    return dot

In [ ]:
display(plot_char_dfa(json_fsm))

In [ ]:
vocab_list = [tokenizer.decode([i]) for i in range(tokenizer.vocab_size)]
display(plot_token_fsm(json_index, vocab_list, json_fsm.initial, json_fsm.finals))